# DATA 304 — Module 9 Assignment: String Processing and Text Normalization

## Q1. Extract course codes from text

**Task**

You are given a `pandas` Series of strings that may contain university course codes, such as `CS101`, `MATH200`, or `bio120`.
Using a **compiled regex pattern** and `Series.str.extract`, create a DataFrame `out_q1` with a single column named `course` containing the **first valid course code** found in each row.

**Rules**

* A valid course code has **2–4 letters** followed by **3 digits** (e.g., `CS101`, `STAT220`, `BIO120`).
* The letters and digits may be separated by whitespace or a single `-`.
* The match should be **case-insensitive**.
* If no valid course code exists in the string, return `NaN`.

In [ ]:
# ASSIGNMENT CELL: Q1
import re
import pandas as pd

s = pd.Series([
    "CS101, MATH200, BIO120",
    "Course: cs-101",
    "Enroll in STAT220 today",
    "Data Science Rocks",
    "New AI  101 is here!",
    "Even Data -  304 is offered."
])

# your code starts here
pat = re.compile(r'(?i)\b([A-Z]{2,4}\s*-?\s*\d{3})\b')
out_q1 = s.str.extract(pat)
out_q1.columns = ['course']
# your code ends here

display(out_q1)

SyntaxError: invalid syntax (1169897605.py, line 15)

## Q2. Exploring greedy vs. non-greedy matches

**Task**

You are given an HTML-like string containing multiple tags of the same type.
Use `re.findall` twice, once with a **greedy** pattern and once with a **non-greedy** pattern to observe how many elements are captured in each case.

Create two variables:

* `greedy_matches` — matches using a greedy pattern
* `nongreedy_matches` — matches using a non-greedy version of the same pattern

**Hints**

* Use the same base pattern `<div>...</div>`.
* Modify only the quantifier so one pattern is greedy and the other is not.
* Focus on understanding how many matches each approach returns, not which one is “correct.”

In [ ]:
# ASSIGNMENT CELL: Q2
import re

html = "<div>first</div><div>second</div><div>third</div>"

# your code starts here
greedy_matches = re.findall(r"<div>.+</div>", html)
nongreedy_matches = re.findall(r"<div>.+?</div>", html)
# your code ends here

display(greedy_matches)
display(nongreedy_matches)

## Q3. Whole-word search using boundaries

**Task**

You are given a `pandas` Series of short text strings.
Your goal is to find all rows that contain the **whole word** `cat` (case-insensitive).
Use a **regular expression with word boundaries** to avoid matching words like `bobcat` or `concatenate`.

Create:

* `mask_q3` — a Boolean Series indicating which rows contain the word `cat`
* `hits_q3` — a filtered Series containing only the matching rows

**Hints**

* Use `Series.str.contains()` with `regex=True`.
* Apply the `case=False` argument for case-insensitivity.
* The word boundary symbol in regex is `\b`.

In [ ]:
# ASSIGNMENT CELL: Q3
import pandas as pd

s = pd.Series([
    "The cat sat on the mat.",
    "Concatenate strings easily.",
    "Bobcat spotted in the park.",
    "A CAT was sleeping.",
    "dog and catfish are unrelated."
])

# your code starts here
mask_q3 = s.str.contains(r"\bcat\b", regex=True, case=False)
hits_q3 = s[mask_q3]
# your code ends here

display(mask_q3)
display(hits_q3)

## Q4. Normalize messy text strings

**Task**

You are given a `pandas` Series of short text snippets that contain inconsistent capitalization, punctuation, accents, and spacing.
Your goal is to create a normalized version of each string by applying a small text-cleaning function.

Create a function `normalize(text)` that performs the following steps in order:

1. Convert the text to lowercase.
2. Remove accents using `unicodedata.normalize("NFKD", text)` and by filtering out combining characters.
3. Remove all punctuation and symbols using `re.sub(r"[^\w\s]", " ", text)`.
4. Replace multiple whitespace with a single space and trim leading/trailing spaces.

Apply this function to the Series and store the cleaned result in `clean_q4`.

**Hints**

* Use `.apply(normalize)` on the Series.
* Keep the output all lowercase and space-separated.

In [ ]:
# ASSIGNMENT CELL: Q4
import re
import unicodedata
import pandas as pd

raw = pd.Series([
    "Résumé: Data—Wrangling!",
    "Naïve café\t visitors...",
    "HELLO!!!\nWorld",
    "Curaçao & Jalapeño"
])

# your code starts here
def strip_accents(t: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", t)
                   if not unicodedata.combining(c))

def normalize(text: str) -> str:
    t = text.lower()
    t = strip_accents(t)
    t = re.sub(r"[^\w\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

clean_q4 = raw.apply(normalize)
# your code ends here

display(clean_q4)

## Q5. Extract all 4-digit years per row

**Task**

You are given a `pandas` Series of text lines, each possibly containing one or more **4-digit years**.
Use a **regular expression** and `Series.str.extractall()` to capture **all 4-digit years** from each row.
Then aggregate the matches back into a single comma-separated string for each original row.

Create a new Series `years_q5` that lists all 4-digit years found in each row, joined by commas.
If a row contains no year, store an empty string (`""`).

**Hints**

* `extractall()` creates a MultiIndex; use `groupby(level=0)` to re-aggregate.
* Use `", ".join(...)` to combine multiple matches per row.

In [2]:
# ASSIGNMENT CELL: Q5
import pandas as pd

text = pd.Series([
    "Hires in 2019 and ramp-up 2020; peak 2021.",
    "Legacy batches 1998 and 2005; refresh 2017, 2017.",
    "No years mentioned here.",
    "Targets: 2024-2026 plan; prior 2010."
])

# your code starts here
matches = text.str.extractall(r"(\b\d{4}\b)")
years_q5 = (
    matches.groupby(level=0)[0]
    .apply(lambda s: ", ".join(s))
    .reindex(text.index)
    .fillna("")
)
# your code ends here

display(years_q5)

0          2019, 2020, 2021
1    1998, 2005, 2017, 2017
2                          
3          2024, 2026, 2010
Name: 0, dtype: object